# SW-10-CSharp-RDFStar — Jumeau C# : annoter des triplets (réification) via dotNetRDF

> **Jumeau C# (.NET 9)** de [`SW-10-Python-RDFStar`](SW-10-Python-RDFStar.ipynb). Le notebook Python utilise **rdflib** pour illustrer l'annotation de triplets (provenance, confiance, validité temporelle) via la **réification classique** RDF (4 triplets par assertion) et les **graphes nommés** (`rdflib.Dataset`). **Ce jumeau invoque le vrai moteur .NET natif** : **dotNetRDF 3.4.1** (`VDS.RDF.Graph`, `Triple`, `SparqlQueryParser`, `ExecuteQuery`) — pas une réimplémentation jouet.

**Source Python** : [`SW-10-Python-RDFStar.ipynb`](SW-10-Python-RDFStar.ipynb) — `rdflib.Graph`/`Dataset`, `reify()` helper, requêtes SPARQL sur `rdf:Statement`.

## Objectifs d'apprentissage

1. **Comprendre la limitation** fondamentale de RDF : un triplet est une affirmation atomique, on ne peut pas (en RDF 1.0) annoter directement un triplet.
2. **Maîtriser la réification classique** : décomposer une assertion en 4 triplets (`rdf:Statement` + `rdf:subject`/`predicate`/`object`) pour pouvoir l'annoter.
3. **Construire un graphe de connaissances annoté** : provenance, score de confiance, validité temporelle.
4. **Interroger** ces annotations via SPARQL (`SparqlQueryParser` + `ExecuteQuery`), avec filtrage.
5. **Découvrir les graphes nommés** comme alternative plus compacte à la réification.

## Vérification mathématique (cibles de concordance)

- **Coût de la réification** : annoter 1 fait = **7 triplets** exactement (1 original + 4 réification + 2 annotations). Concordance avec rdflib.
- **Graphe de connaissances** : 4 personnes (8 triplets `type`/`name`) + faits annotés.
- **Requête SPARQL filtrée** : 2 relations à confiance ≥ 0,60 (Alice→Bob 0,95, Bob→Carol 0,60), 2 à confiance < 0,60 (Carol→Dave 0,30, Alice→Dave 0,50). Concordance avec le twin Python.


## 1. Installation et espace de noms

dotNetRDF est le moteur .NET natif pour RDF/SPARQL. On charge le NuGet `dotNetRDF` 3.4.1 (le même que `SW-9-CSharp-JSONLD`). On définit les préfixes et URI de base comme le twin Python (`ex:`, `foaf:`, `rdf:`, `xsd:`).

In [1]:
#r "nuget: dotNetRDF, 3.4.1"

using VDS.RDF;
using VDS.RDF.Parsing;
using VDS.RDF.Query;
using VDS.RDF.Writing;
using System;
using System.IO;
using System.Text;

Console.WriteLine("dotNetRDF charge (VDS.RDF) -- marathon #4956, parite avec SW-10-Python-RDFStar (rdflib)");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages dotNetRDF, 3.4.1

dotNetRDF charge (VDS.RDF) -- marathon #4956, parite avec SW-10-Python-RDFStar (rdflib)


## 2. Le problème : pourquoi annoter des triplets ?

RDF représente des faits sous forme de **triplets** (sujet, prédicat, objet). Mais un triplet est atomique : on ne peut pas lui attacher directement une provenance, un score de confiance ou une date. Pour dire « Alice affirme que Bob connaît Carol avec 85 % de confiance », il faut **réifier** le triplet — le décomposer en 4 triplets auxiliaires — puis annoter ce nœud-réification.

### 2.1 La réification classique : 4 triplets pour 1 assertion

La réification d'un triplet `(s, p, o)` crée un nœud `stmt` typé `rdf:Statement` relié au triplet original via `rdf:subject`, `rdf:predicate`, `rdf:object`. On peut alors annoter `stmt` (source, confiance, date) sans toucher au fait original.

In [2]:
// Espace de noms et URI de base (identiques au twin Python)
const string NS = "http://example.org/";
var g = new Graph();
g.NamespaceMap.AddNamespace("ex", new Uri(NS));
g.NamespaceMap.AddNamespace("foaf", new Uri("http://xmlns.com/foaf/0.1/"));

// Helper : un URI node racourci
IUriNode U(string local) => g.CreateUriNode("ex:" + local);
IUriNode Pred(Uri u) => g.CreateUriNode(u);
IUriNode rdfType = g.CreateUriNode("rdf:type");
IUriNode rdfStmt = g.CreateUriNode("rdf:Statement");
IUriNode rdfSubject = g.CreateUriNode("rdf:subject");
IUriNode rdfPredicate = g.CreateUriNode("rdf:predicate");
IUriNode rdfObject = g.CreateUriNode("rdf:object");
IUriNode foafKnows = Pred(new Uri("http://xmlns.com/foaf/0.1/knows"));

// Reification classique : decomposer le triplet (Bob, knows, Carol)
g.Assert(new Triple(U("Bob"), foafKnows, U("Carol")));          // le fait original (1)

var stmt = U("statement1");
g.Assert(new Triple(stmt, rdfType, rdfStmt));                   // rdf:type rdf:Statement
g.Assert(new Triple(stmt, rdfSubject, U("Bob")));               // rdf:subject Bob
g.Assert(new Triple(stmt, rdfPredicate, foafKnows));            // rdf:predicate foaf:knows
g.Assert(new Triple(stmt, rdfObject, U("Carol")));              // rdf:object Carol   (4)

// Annotations : qui affirme ce fait, avec quel degre de confiance ?
g.Assert(new Triple(stmt, U("assertedBy"), U("Alice")));        // assertedBy Alice
g.Assert(new Triple(stmt, U("confidence"), g.CreateLiteralNode("0.85", new Uri("http://www.w3.org/2001/XMLSchema#double"))));  // (2)

Console.WriteLine($"Nombre total de triplets : {g.Triples.Count}");
Console.WriteLine("  - Fait original : 1 triplet");
Console.WriteLine("  - Reification : 4 triplets");
Console.WriteLine("  - Annotations : 2 triplets");
Console.WriteLine("  - Total : 7 triplets pour annoter 1 fait");


Nombre total de triplets : 7


  - Fait original : 1 triplet


  - Reification : 4 triplets


  - Annotations : 2 triplets


  - Total : 7 triplets pour annoter 1 fait


### Interprétation : coût de la réification

| Approche | Triplets nécessaires | Lisibilité | Garantie d'existence du fait |
|----------|---------------------|------------|------------------------------|
| Triplet nu | 1 | excellente | oui (le triplet existe) |
| Réification + annotations | **7** | verbeuse | non (le fait original peut ne pas exister) |

C'est précisément ce coût (4 triplets auxiliaires + N annotations) que **RDF-star** (RDF 1.2) vise à éliminer avec la syntaxe `<< s p o >>`. Mais la réification reste le mécanisme portable, indépendant du moteur. La sérialisation Turtle ci-dessous montre les 7 triplets.

In [3]:
// Serialisation Turtle (equivalent rdflib.serialize(format="turtle"))
var writer = new CompressingTurtleWriter();
var sw = new System.IO.StringWriter();
writer.Save(g, sw);
Console.WriteLine(sw.ToString());


@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>.
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>.
@prefix xsd: <http://www.w3.org/2001/XMLSchema#>.
@prefix ex: <http://example.org/>.
@prefix foaf: <http://xmlns.com/foaf/0.1/>.

ex:Bob foaf:knows ex:Carol.
ex:statement1 ex:assertedBy ex:Alice;
              ex:confidence "0.85"^^xsd:double;
              rdf:object ex:Carol;
              rdf:predicate foaf:knows;
              rdf:subject ex:Bob;
              a rdf:Statement.



## 3. Fonction utilitaire `Reify()`

Pour réduire la verbosité, le twin Python définit une fonction `reify(graph, s, p, o)`. Voici l'équivalent C# : elle crée le nœud-réification `rdf:Statement` + les 3 liens, sans répéter le fait original (à ajouter séparément). Retourne le nœud `stmt` pour annotation.

In [4]:
// Reify : cree les 4 triplets de reification et retourne le noeud statement.
// (Le fait original doit etre ajoute separement si on veut qu'il existe dans le graphe.)
IUriNode Reify(IUriNode s, IUriNode p, IUriNode o, string name)
{
    var stmtNode = U(name);
    g.Assert(new Triple(stmtNode, rdfType, rdfStmt));
    g.Assert(new Triple(stmtNode, rdfSubject, s));
    g.Assert(new Triple(stmtNode, rdfPredicate, p));
    g.Assert(new Triple(stmtNode, rdfObject, o));
    return stmtNode;
}

// Helper pour creer un LiteralNode de confiance typé xsd:double.
ILiteralNode Confidence(double v) =>
    g.CreateLiteralNode(v.ToString(System.Globalization.CultureInfo.InvariantCulture),
                         new Uri("http://www.w3.org/2001/XMLSchema#double"));

// Demo : un 2e fait reifie via le helper (Bob connait Dave, affirmé par Carol).
g.Assert(new Triple(U("Bob"), foafKnows, U("Dave")));              // fait original
var stmt2 = Reify(U("Bob"), foafKnows, U("Dave"), "statement2");
g.Assert(new Triple(stmt2, U("assertedBy"), U("Carol")));
g.Assert(new Triple(stmt2, U("confidence"), Confidence(0.60)));

Console.WriteLine($"Apres 2e fait reifie : graphe g = {g.Triples.Count} triplets au total");


Apres 2e fait reifie : graphe g = 14 triplets au total


## 4. Graphe de connaissances annoté

Construisons un graphe réaliste : 4 personnes (Alice, Bob, Carol, Dave) reliées par `foaf:knows`, chaque relation annotée d'un score de confiance et d'une source. Le fait 4 (Alice connaît Dave) est **seulement affirmé par Bob** mais n'existe pas comme triplet direct — c'est précisément l'intérêt de la réification : on peut enregistrer une affirmation sans l'ajouter au graphe des faits.

In [5]:
var gk = new Graph();
gk.NamespaceMap.AddNamespace("ex", new Uri(NS));
gk.NamespaceMap.AddNamespace("foaf", new Uri("http://xmlns.com/foaf/0.1/"));

// Helpers locaux (gk)
IUriNode N(string local) => gk.CreateUriNode("ex:" + local);
IUriNode foafName = gk.CreateUriNode(UriFactory.Create("http://xmlns.com/foaf/0.1/name"));
IUriNode rdfTypeK = gk.CreateUriNode("rdf:type");
IUriNode foafPerson = gk.CreateUriNode(UriFactory.Create("http://xmlns.com/foaf/0.1/Person"));
IUriNode knowsK = gk.CreateUriNode(UriFactory.Create("http://xmlns.com/foaf/0.1/knows"));
ILiteralNode Str(string s) => gk.CreateLiteralNode(s);
ILiteralNode Conf(double v) => gk.CreateLiteralNode(
    v.ToString(System.Globalization.CultureInfo.InvariantCulture),
    new Uri("http://www.w3.org/2001/XMLSchema#double"));

// Reify sur gk : retourne le noeud statement
IUriNode ReifyK(IUriNode s, IUriNode p, IUriNode o, string name)
{
    var st = N(name);
    gk.Assert(new Triple(st, rdfTypeK, gk.CreateUriNode("rdf:Statement")));
    gk.Assert(new Triple(st, gk.CreateUriNode("rdf:subject"), s));
    gk.Assert(new Triple(st, gk.CreateUriNode("rdf:predicate"), p));
    gk.Assert(new Triple(st, gk.CreateUriNode("rdf:object"), o));
    return st;
}

// === Personnes ===
foreach (var (name, uri) in new[] { ("Alice Dupont", N("Alice")), ("Bob Martin", N("Bob")),
                                     ("Carol Laurent", N("Carol")), ("Dave Petit", N("Dave")) })
{
    gk.Assert(new Triple(uri, rdfTypeK, foafPerson));
    gk.Assert(new Triple(uri, foafName, Str(name)));
}

// === Fait 1 : Alice connait Bob (haute confiance, LinkedIn) ===
gk.Assert(new Triple(N("Alice"), knowsK, N("Bob")));
var s1 = ReifyK(N("Alice"), knowsK, N("Bob"), "stmt1");
gk.Assert(new Triple(s1, N("confidence"), Conf(0.95)));
gk.Assert(new Triple(s1, N("source"), Str("LinkedIn")));

// === Fait 2 : Bob connait Carol (confiance moyenne, Twitter) ===
gk.Assert(new Triple(N("Bob"), knowsK, N("Carol")));
var s2 = ReifyK(N("Bob"), knowsK, N("Carol"), "stmt2");
gk.Assert(new Triple(s2, N("confidence"), Conf(0.60)));
gk.Assert(new Triple(s2, N("source"), Str("Twitter")));

// === Fait 3 : Carol connait Dave (basse confiance, rumeur) ===
gk.Assert(new Triple(N("Carol"), knowsK, N("Dave")));
var s3 = ReifyK(N("Carol"), knowsK, N("Dave"), "stmt3");
gk.Assert(new Triple(s3, N("confidence"), Conf(0.30)));
gk.Assert(new Triple(s3, N("source"), Str("Rumeur")));

// === Fait 4 : Alice connait Dave (NON asserte, seulement affirme par Bob) ===
var s4 = ReifyK(N("Alice"), knowsK, N("Dave"), "stmt4");
gk.Assert(new Triple(s4, N("assertedBy"), N("Bob")));
gk.Assert(new Triple(s4, N("confidence"), Conf(0.50)));

Console.WriteLine($"Graphe de connaissances : {gk.Triples.Count} triplets");


Graphe de connaissances : 35 triplets


## 5. Interroger les annotations avec SPARQL

Pour extraire les relations annotées, on joint sur `rdf:subject`/`rdf:predicate`/`rdf:object` et on récupère les annotations. La requête ci-dessous liste toutes les relations connues avec leur confiance et source, triées par confiance décroissante.

In [6]:
// Requete 1 : toutes les relations annotées (joiture sur rdf:Statement)
string queryAll = @"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex:   <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?person1 ?person2 ?confidence ?source
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?person1 ;
          rdf:predicate foaf:knows ;
          rdf:object ?person2 ;
          ex:confidence ?confidence .
    OPTIONAL { ?stmt ex:source ?source . }
}
ORDER BY DESC(?confidence)";

// Helper : extraire la valeur lexicale d'un LiteralNode (sans le suffixe ^^datatype).
// r[v].ToString() renvoie "0.95^^...XSD#double" -> on veut "0.95".
double ConfVal(ISparqlResult r, string v) =>
    double.Parse(((ILiteralNode)r[v]).Value, System.Globalization.CultureInfo.InvariantCulture);
// Helper : valeur lexicale d'un LiteralNode string (sans le suffixe ^^datatype).
string LitStr(ISparqlResult r, string v) => r.HasValue(v) && r[v] is ILiteralNode ln ? ln.Value : "-";

var parser = new SparqlQueryParser();
var q = parser.ParseFromString(queryAll);
var results = gk.ExecuteQuery(q) as SparqlResultSet;

Console.WriteLine("=== Relations annotées (triées par confiance) ===");
Console.WriteLine($"{"Personne 1",-12} {"Personne 2",-12} {"Confiance",10} {"Source",-12}");
Console.WriteLine(new string('-', 50));
foreach (var r in results)
{
    var p1 = r["person1"].ToString().Split('/').Last();
    var p2 = r["person2"].ToString().Split('/').Last();
    var conf = ConfVal(r, "confidence");
    var src = LitStr(r, "source");
    Console.WriteLine($"{p1,-12} {p2,-12} {conf,10:F2} {src,-12}");
}


=== Relations annotées (triées par confiance) ===


Personne 1   Personne 2    Confiance Source      


--------------------------------------------------


Alice        Bob                0,95 LinkedIn    


Bob          Carol              0,60 Twitter     


Alice        Dave               0,50 -           


Carol        Dave               0,30 Rumeur      


## 6. Filtrer par score de confiance

Cas d'usage clé : séparer les **faits fiables** (confiance ≥ 0,60) des **faits incertains** (< 0,60). On utilise une clause `FILTER` SPARQL.

In [7]:
string queryHigh = @"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex:   <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

SELECT ?person1 ?person2 ?confidence ?source
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?person1 ;
          rdf:predicate foaf:knows ;
          rdf:object ?person2 ;
          ex:confidence ?confidence .
    FILTER (?confidence >= ""0.60""^^xsd:double)
    OPTIONAL { ?stmt ex:source ?source . }
}
ORDER BY DESC(?confidence)";

string queryLow = @"
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ex:   <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

SELECT ?person1 ?person2 ?confidence ?source
WHERE {
    ?stmt a rdf:Statement ;
          rdf:subject ?person1 ;
          rdf:predicate foaf:knows ;
          rdf:object ?person2 ;
          ex:confidence ?confidence .
    FILTER (?confidence < ""0.60""^^xsd:double)
    OPTIONAL { ?stmt ex:source ?source . }
}
ORDER BY ?confidence";

var rsHigh = (SparqlResultSet)gk.ExecuteQuery(new SparqlQueryParser().ParseFromString(queryHigh));
var rsLow  = (SparqlResultSet)gk.ExecuteQuery(new SparqlQueryParser().ParseFromString(queryLow));

Console.WriteLine($"=== Relations FIABLES (confiance >= 0.60) : {rsHigh.Count} ===");
foreach (var r in rsHigh)
{
    var p1 = r["person1"].ToString().Split('/').Last();
    var p2 = r["person2"].ToString().Split('/').Last();
    var src = LitStr(r, "source");
    var conf = ConfVal(r, "confidence");
    Console.WriteLine($"  {p1} connait {p2} (confiance={conf:F2}, source={src})");
}

Console.WriteLine();
Console.WriteLine($"=== Relations INCERTAINES (confiance < 0.60) : {rsLow.Count} ===");
foreach (var r in rsLow)
{
    var p1 = r["person1"].ToString().Split('/').Last();
    var p2 = r["person2"].ToString().Split('/').Last();
    var src = LitStr(r, "source");
    var conf = ConfVal(r, "confidence");
    Console.WriteLine($"  {p1} connait {p2} (confiance={conf:F2}, source={src})");
}


=== Relations FIABLES (confiance >= 0.60) : 2 ===


  Alice connait Bob (confiance=0,95, source=LinkedIn)


  Bob connait Carol (confiance=0,60, source=Twitter)


=== Relations INCERTAINES (confiance < 0.60) : 2 ===


  Carol connait Dave (confiance=0,30, source=Rumeur)


  Alice connait Dave (confiance=0,50, source=-)


## 7. Alternative : les graphes nommés

Les **graphes nommés** (Named Graphs) offrent une autre approche pour grouper des triplets par contexte : plutôt que de réifier chaque triplet individuellement, on regroupe un ensemble de triplets sous une URI de graphe. On peut alors annoter le graphe tout entier (provenance, confiance globale). dotNetRDF expose cela via la classe `VDS.RDF.ThreadSafeTripleStore` + `IGraph` nommés.

In [8]:
var store = new TripleStore();

// dotNetRDF 3.x : un graphe nomme se construit via Graph(IRefNode) -- le noeud-nom
// (Name, IRefNode) definit l'identite du graphe dans le store. Le vieux pattern
// `new Graph { BaseUri = uri }` laisse Name == null et les graphes se confondent
// avec le graphe par defaut -> collision a l'ajout. On garde aussi un guard HasGraph
// (non deprecate, via IRefNode) pour rester idempotent sur re-execution dans l'IDE.
var nLinkedIn = new Graph().CreateUriNode(UriFactory.Create(NS + "graph/linkedin"));
var gLinkedIn = new Graph(nLinkedIn);
gLinkedIn.Assert(new Triple(
    gLinkedIn.CreateUriNode(UriFactory.Create(NS + "Alice")),
    gLinkedIn.CreateUriNode(UriFactory.Create("http://xmlns.com/foaf/0.1/knows")),
    gLinkedIn.CreateUriNode(UriFactory.Create(NS + "Bob"))));
if (!store.HasGraph(nLinkedIn)) store.Add(gLinkedIn);

var nTwitter = new Graph().CreateUriNode(UriFactory.Create(NS + "graph/twitter"));
var gTwitter = new Graph(nTwitter);
gTwitter.Assert(new Triple(
    gTwitter.CreateUriNode(UriFactory.Create(NS + "Bob")),
    gTwitter.CreateUriNode(UriFactory.Create("http://xmlns.com/foaf/0.1/knows")),
    gTwitter.CreateUriNode(UriFactory.Create(NS + "Carol"))));
if (!store.HasGraph(nTwitter)) store.Add(gTwitter);

// dotNetRDF 3.x : les requetes sur store passent par un ISparqlQueryProcessor.
// On liste ici directement les graphes nommes du store et leur contenu (plus pedagogique).
Console.WriteLine("=== Graphes nommés du store ===");
foreach (IGraph ng in store.Graphs)
{
    var gName = ng.Name is IUriNode un ? un.Uri.ToString().Split('/').Last() : "(default)";
    Console.WriteLine($"  graphe {gName} : {ng.Triples.Count} triplet(s)");
    foreach (var t in ng.Triples)
        Console.WriteLine($"    {t.Subject} {t.Predicate} {t.Object}");
}


=== Graphes nommés du store ===


  graphe linkedin : 1 triplet(s)


    http://example.org/Alice http://xmlns.com/foaf/0.1/knows http://example.org/Bob


  graphe twitter : 1 triplet(s)


    http://example.org/Bob http://xmlns.com/foaf/0.1/knows http://example.org/Carol


## 8. Sérialisation et interopérabilité

dotNetRDF supporte plusieurs formats de sérialisation (Turtle, N-Triples, RDF/XML, JSON-LD). Le Turtle (compressé) est le plus lisible pour la réification.

In [9]:
// Serialisation du graphe de connaissances en Turtle compressé
var w = new CompressingTurtleWriter();
var sw2 = new System.IO.StringWriter();
w.Save(gk, sw2);
var turtle = sw2.ToString();
Console.WriteLine($"Serialisation Turtle : {turtle.Length} caracteres, {gk.Triples.Count} triplets");
Console.WriteLine("--- (extrait) ---");
Console.WriteLine(turtle.Substring(0, Math.Min(500, turtle.Length)));


Serialisation Turtle : 1370 caracteres, 35 triplets


--- (extrait) ---


@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>.
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>.
@prefix xsd: <http://www.w3.org/2001/XMLSchema#>.
@prefix ex: <http://example.org/>.
@prefix foaf: <http://xmlns.com/foaf/0.1/>.

ex:Alice a foaf:Person;
         foaf:knows ex:Bob;
         foaf:name "Alice Dupont".
ex:Bob a foaf:Person;
       foaf:knows ex:Carol;
       foaf:name "Bob Martin".
ex:Carol a foaf:Person;
         foaf:knows ex:Dave;
         foaf:name "


## Exercices

Les exercices ci-dessous sont à compléter (convention C.1 : stubs exécutables sans erreur).

### Exercice 1 — Ajouter une annotation de date de validité
Étendez le graphe de connaissances : pour chaque fait annoté, ajoutez un triplet `ex:validSince` avec une date typée `xsd:date`. Modifiez ensuite la requête SPARQL pour afficher cette colonne.

In [10]:
// Exercice 1 : annotation temporelle
// TODO : pour chaque stmt (stmt1..stmt4), ajouter gk.Assert(new Triple(stmt, N("validSince"),
//        gk.CreateLiteralNode("2020-01-01", xsd:date))).
// Indice : utilisez gk.CreateLiteralNode(dateStr, new Uri("http://www.w3.org/2001/XMLSchema#date")).
// Etape 1 : choisir une date de validité par fait. Etape 2 : modifier queryAll pour ajouter ?since.
Console.WriteLine("Exercice 1 a completer : annotation de date de validite sur les faits.");


Exercice 1 a completer : annotation de date de validite sur les faits.


### Exercice 2 — Requête de transitivité inférée
Alice connaît Bob (0,95), Bob connaît Carol (0,60). Écrivez une requête SPARQL qui calcule les « connaissances transitives » (chemins de longueur 2) et combine les confiances par produit.

In [11]:
// Exercice 2 : transitivité des connaissances
// TODO : requete SPARQL avec deux jointures ?a foaf:knows ?b . ?b foaf:knows ?c .
// Indice : la confiance d'un chemin a->b->c = confiance(a->b) * confiance(b->c).
// Etape 1 : ecrire la jointure sur les stmt reifies. Etape 2 : multiplier les confiances dans le SELECT.
Console.WriteLine("Exercice 2 a completer : requete de connaissances transitives (longueur 2).");


Exercice 2 a completer : requete de connaissances transitives (longueur 2).


### Exercice 3 — Graphes nommés multi-sources
Construisez un 3e graphe nommé (par ex. `graph/intranet`) avec un fait Bob→Dave, puis écrivez une requête qui fusionne les 3 graphes et identifie les faits présents dans plusieurs sources.

In [12]:
// Exercice 3 : fusion multi-sources via graphes nommés
// TODO : ajouter store.Add(gIntranet) avec un fait, puis requete UNION/GRAPH combinant les sources.
// Indice : une requete SELECT ?g ?s ?o WHERE { GRAPH ?g { ?s foaf:knows ?o } } liste tous les faits
// avec leur graphe d'origine ; un GROUP BY ?s ?o HAVING(COUNT(*)>1) trouve les faits multi-sources.
Console.WriteLine("Exercice 3 a completer : 3e graphe nommé + detection des faits multi-sources.");


Exercice 3 a completer : 3e graphe nommé + detection des faits multi-sources.


## 9. Conclusion — synthèse

| Mécanisme | Coût (triplets) | Lisibilité | Portabilité |
|-----------|----------------|------------|-------------|
| Triplet nu | 1 | excellente | totale |
| Réification classique | 4 + N annotations | verbeuse | totale (RDF 1.0) |
| RDF-star (`<< >>`) | 1 + N | excellente | récente (RDF 1.2) |
| Graphes nommés | 1 + annotation du graphe | bonne | totale (SPARQL 1.1) |

**Leçon clé** : la **réification classique** est le mécanisme portable pour annoter des triplets (provenance, confiance, temporalité) — au prix d'une verbosité de 4 triplets auxiliaires par fait. RDF-star (RDF 1.2) vise à éliminer ce coût avec une syntaxe dédiée `<< s p o >>`, mais reste un standard jeune. Les **graphes nommés** offrent un compromis intermédiaire : grouper des triplets par contexte et annoter le groupe. dotNetRDF gère les trois (Graph, SparqlQueryParser, TripleStore) avec la même API `VDS.RDF` que ce notebook utilise.

---

**Navigation** : [`<< SW-9 JSONLD`](SW-9-CSharp-JSONLD.ipynb) | [Index SemanticWeb](README.md) | Jumeau Python : [`SW-10-Python-RDFStar`](SW-10-Python-RDFStar.ipynb)

*Marathon #4956 — parité .NET ⇄ Python. Prong A : vrai moteur dotNetRDF 3.4.1 natif invoqué.*
